# Lab 2.3 — REINFORCE: Monte Carlo Policy Gradient with PyTorch
**Course:** AI Agents · **Unit:** UC2 2.4 Policy Gradient Methods · **Environment:** `CartPole-v1`

In the lecture we saw that a **policy-based** agent learns the policy $\pi_\theta(a \mid s)$ directly, and that the **Policy Gradient Theorem** lets us estimate
$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}\big[\nabla_\theta \log \pi_\theta(a_t \mid s_t)\, G_t\big]$$
from sampled episodes. In this lab you will implement **REINFORCE** (slide 16) with the **simple policy network** from slide 20 and train it to balance a pole.

### Learning objectives
By the end of this lab you will be able to:
1. Build a stochastic policy network that outputs a probability distribution over actions.
2. Compute the discounted returns $G_t$ of an episode with a backward pass.
3. Turn the policy gradient into a loss that a PyTorch optimizer can minimize.
4. Implement and train the full REINFORCE loop, and explain its high variance.

### How to work
- Cells marked **`TODO n`** contain `raise NotImplementedError(...)`. Replace that line with your code.
- After each part there is a **check cell**. Run it: it must print ✅ before you move on.
- Answer the questions in Part 8 in the markdown cells provided.

| Part | Topic | Slides |
|---|---|---|
| 0 | Setup | – |
| 1 | Meet CartPole | 6 |
| 2 | Policy network | 6, 20 |
| 3 | Discounted returns $G_t$ | 17 |
| 4 | The REINFORCE loss | 19 |
| 5 | The REINFORCE algorithm | 16 |
| 6 | Training and learning curve | 21 |
| 7 | Evaluation | 18 |
| 8 | Experiments and questions | 18, 21 |

## Part 0 — Setup
If you work in Google Colab or a fresh environment, uncomment and run the install line first.

In [ ]:
# %pip install -q gymnasium torch matplotlib

In [ ]:
import os

os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')


import random
from collections import deque

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed()
device = torch.device("cpu")   # a tiny network on CartPole is faster on CPU than on GPU
print("PyTorch", torch.__version__, "| Gymnasium", gym.__version__)

## Part 1 — Meet CartPole (slide 6)
A pole is attached to a cart moving on a track. The agent pushes the cart **left (0)** or **right (1)**.
- **State** $s_t$ (4 numbers): cart position, cart velocity, pole angle, pole angular velocity.
- **Reward:** +1 for every step the pole stays up.
- **Episode ends** when the pole tilts more than 12°, the cart leaves the track, or after **500 steps** (maximum return = 500).

### TODO 1
1. Create the environment `CartPole-v1` and store it in `env`.
2. Print its `observation_space` and `action_space`, and store the sizes in `state_size` and `action_size`.
3. Play **one** episode with a **random** policy (`env.action_space.sample()`) and print its length.

In [ ]:
# ---------------------------- TODO 1 ----------------------------
# YOUR CODE HERE
raise NotImplementedError("TODO 1: create the environment and play a random episode")
# ----------------------------------------------------------------

In [ ]:
# ✔ Check Part 1
assert state_size == 4 and action_size == 2, "CartPole has 4 state variables and 2 actions"
print("✅ Part 1 OK — a random agent is our baseline: it usually survives only 10–40 steps.")

## Part 2 — The policy network (slides 6 and 20)
Our policy $\pi_\theta(a \mid s)$ is a neural network with parameters $\theta$. It receives the state and outputs **one probability per action** (they sum to 1).

Architecture (slide 20):

| Layer | Size | Activation |
|---|---|---|
| Input | `state_size` = 4 | – |
| Hidden 1 | 64 | ReLU |
| Hidden 2 | 64 | ReLU |
| Hidden 3 | 64 | ReLU |
| Output | `action_size` = 2 | **Softmax** |

To **act**, we do not pick the most probable action: we **sample** from the distribution (slide 18 — this is where exploration comes from) and we keep $\log \pi_\theta(a_t \mid s_t)$ because the update rule needs it.

### TODO 2 — define the layers in `__init__`
### TODO 3 — `forward`: three hidden layers with ReLU, then softmax over the actions
### TODO 4 — `act`: convert the state to a tensor, get the probabilities, build a `Categorical`, sample an action, return `(action as int, log_prob)`

> 💡 Hints: `nn.Linear(in, out)`, `F.relu`, `F.softmax(x, dim=-1)`, `torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)` adds the batch dimension, `dist = Categorical(probs)`, `dist.sample()`, `dist.log_prob(action)`. **Do not** wrap `act` in `torch.no_grad()`: the log-probability must keep its gradient.

In [ ]:
class PolicyNetwork(nn.Module):
    def __init__(self, state_size, action_size, hidden_size=64):
        super().__init__()
        # ---------------------------- TODO 2 ----------------------------
        # Three hidden fully connected layers of `hidden_size` units and
        # one output layer with `action_size` units (slide 20).
        # YOUR CODE HERE
        raise NotImplementedError("TODO 2: define the layers")
        # ----------------------------------------------------------------

    def forward(self, x):
        # ---------------------------- TODO 3 ----------------------------
        # x has shape (batch, state_size). Return action probabilities
        # with shape (batch, action_size).
        # YOUR CODE HERE
        raise NotImplementedError("TODO 3: forward pass with ReLU and softmax")
        # ----------------------------------------------------------------

    def act(self, state):
        # ---------------------------- TODO 4 ----------------------------
        # state: numpy array of shape (state_size,)
        # return: action (int), log_prob (0-dim tensor that keeps its gradient)
        # YOUR CODE HERE
        raise NotImplementedError("TODO 4: sample an action and return its log-probability")
        # ----------------------------------------------------------------

In [ ]:
# ✔ Check Part 2
set_seed()
_policy = PolicyNetwork(4, 2).to(device)
_p = _policy(torch.randn(5, 4))
assert _p.shape == (5, 2), f"forward must return (batch, 2), got {tuple(_p.shape)}"
assert torch.allclose(_p.sum(dim=1), torch.ones(5)), "each row of probabilities must sum to 1"
_n_params = sum(p.numel() for p in _policy.parameters())
assert _n_params == 8770, f"expected 8770 parameters for 4-64-64-64-2, got {_n_params}"
_a, _lp = _policy.act(np.zeros(4, dtype=np.float32))
assert isinstance(_a, int) and _a in (0, 1), "act must return the action as a Python int"
assert torch.is_tensor(_lp) and _lp.dim() == 0, "log_prob must be a 0-dim tensor (use .squeeze())"
assert _lp.requires_grad, "log_prob lost its gradient — do not use no_grad/detach/item in act()"
print(f"✅ Part 2 OK — {_n_params} parameters; π(·|s) for a random state: {_p[0].detach().numpy().round(3)}")

## Part 3 — Discounted returns $G_t$ (slide 17)
Line 4 of the algorithm computes, for **every** time step, the discounted sum of the rewards that came **after** it:
$$G_t = \sum_{k=t}^{T-1} \gamma^{\,k-t}\, r_k$$
Computing this sum separately for each $t$ is $O(T^2)$. Instead, go **backwards** with the recursion
$$G_{T-1} = r_{T-1}, \qquad G_t = r_t + \gamma\, G_{t+1}.$$

**Normalization.** In practice we often standardize the returns of the episode, $\hat G_t = (G_t - \text{mean}) / (\text{std} + 10^{-9})$. It keeps the step size stable and acts like a simple baseline (slide 21).

### TODO 5
Implement `compute_returns(rewards, gamma, normalize=False)` returning a `float32` tensor with one $G_t$ per step.

In [ ]:
def compute_returns(rewards, gamma, normalize=False):
    # ---------------------------- TODO 5 ----------------------------
    # rewards: list of floats [r_0, ..., r_{T-1}]
    # return : torch.FloatTensor [G_0, ..., G_{T-1}]  (standardized if normalize=True)
    # YOUR CODE HERE
    raise NotImplementedError("TODO 5: compute the discounted returns backwards")
    # ----------------------------------------------------------------

In [ ]:
# ✔ Check Part 3 — the worked example of slide 17: 5 steps, reward +1, γ = 0.9
_G = compute_returns([1.0] * 5, gamma=0.9)
print("G_t =", _G.numpy().round(2))
assert torch.allclose(_G, torch.tensor([4.0951, 3.439, 2.71, 1.9, 1.0]), atol=1e-3), "returns do not match slide 17"
_Gn = compute_returns([1.0] * 5, gamma=0.9, normalize=True)
assert abs(_Gn.mean().item()) < 1e-5 and abs(_Gn.std().item() - 1) < 1e-4, "normalized returns need mean 0 and std 1"
assert torch.allclose(compute_returns([0, 0, 1, -1], gamma=0.5), torch.tensor([0.125, 0.25, 0.5, -1.0])), "check a reward sequence that is not constant"
print("✅ Part 3 OK — early actions get credit for everything that followed them.")

## Part 4 — The REINFORCE loss (slide 19)
The update of slide 16 is a gradient **ascent** step: $\theta \leftarrow \theta + \alpha\, G_t\, \nabla_\theta \log \pi_\theta(a_t \mid s_t)$.

PyTorch optimizers **minimize**, so we give them the negative of the quantity we want to maximize, summed over the episode:
$$L(\theta) = -\sum_{t=0}^{T-1} \log \pi_\theta(a_t \mid s_t)\; G_t$$
Calling `loss.backward()` then computes exactly $-\hat g$, and `optimizer.step()` moves $\theta$ uphill on $J(\theta)$.

### TODO 6
Implement `reinforce_loss(log_probs, returns)`.
- `log_probs`: Python **list** of 0-dim tensors (one per step, from `policy.act`).
- `returns`: tensor of shape `(T,)` from `compute_returns`.

> 💡 `torch.stack(list_of_tensors)` turns the list into a tensor of shape `(T,)`.

In [ ]:
def reinforce_loss(log_probs, returns):
    # ---------------------------- TODO 6 ----------------------------
    # YOUR CODE HERE
    raise NotImplementedError("TODO 6: policy-gradient loss")
    # ----------------------------------------------------------------

In [ ]:
# ✔ Check Part 4
_lp = [torch.tensor(np.log(0.5), requires_grad=True), torch.tensor(np.log(0.25), requires_grad=True)]
_loss = reinforce_loss(_lp, torch.tensor([1.0, 2.0]))
assert _loss.dim() == 0, "the loss must be a scalar"
assert abs(_loss.item() - 3.4657) < 1e-3, f"expected 3.4657, got {_loss.item():.4f} — check the minus sign"
_loss.backward()
assert _lp[1].grad.item() == -2.0, "∂L/∂log π(a_t|s_t) should be −G_t"
print("✅ Part 4 OK — ∂L/∂log π = −G_t: minimizing L raises the log-probability of actions with high return.")

## Part 5 — The REINFORCE algorithm (slide 16)
Now put everything together. For each of the `n_episodes` iterations (line 1):

| Pseudocode line | What to do here |
|---|---|
| **2** — generate an episode following $\pi_\theta$ | `TODO 7`: reset the env, then loop: `policy.act(state)` → `env.step(action)`; store `log_prob` and `reward`; stop when `terminated or truncated` (or after `max_t` steps) |
| **4** — compute $G_t$ for all $t$ | `TODO 8`: call `compute_returns` |
| **5** — update $\theta$ | `TODO 9`: compute the loss, then `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()` |

Notice that the update happens **once the episode is over** — that is what makes it Monte Carlo — and that the episode is then thrown away (on-policy).

> 💡 The textbook update of slide 16 also contains a factor $\gamma^t$. Most implementations (including Hugging Face's) drop it; you can test it in the optional TODO 12.

In [ ]:
def reinforce(env, policy, optimizer, n_episodes=1000, gamma=0.99, max_t=500,
              normalize=True, print_every=100):
    scores = []                          # return of every episode
    scores_window = deque(maxlen=100)    # last 100 returns, for the running average

    for i_episode in range(1, n_episodes + 1):
        # ---------------------------- TODO 7 ----------------------------
        # Line 2: generate one episode with the current policy.
        # Fill the lists `log_probs` and `rewards`.
        log_probs, rewards = [], []
        # YOUR CODE HERE
        raise NotImplementedError("TODO 7: generate an episode")
        # ----------------------------------------------------------------

        episode_return = sum(rewards)
        scores.append(episode_return)
        scores_window.append(episode_return)

        # ---------------------------- TODO 8 ----------------------------
        # Line 4: discounted returns G_t for every step of the episode.
        # YOUR CODE HERE
        raise NotImplementedError("TODO 8: compute the returns")
        # ----------------------------------------------------------------

        # ---------------------------- TODO 9 ----------------------------
        # Line 5: one gradient step on the REINFORCE loss.
        # YOUR CODE HERE
        raise NotImplementedError("TODO 9: update the policy")
        # ----------------------------------------------------------------

        if i_episode % print_every == 0:
            print(f"Episode {i_episode:5d} | average return (last 100): {np.mean(scores_window):7.2f}")

    return scores

In [ ]:
# ✔ Check Part 5 — a short smoke test (5 episodes): the code runs and θ changes
set_seed()
_env = gym.make("CartPole-v1")
_env.reset(seed=SEED)
_policy = PolicyNetwork(4, 2).to(device)
_before = [p.detach().clone() for p in _policy.parameters()]
_scores = reinforce(_env, _policy, optim.Adam(_policy.parameters(), lr=1e-3), n_episodes=5, print_every=5)
assert len(_scores) == 5, "reinforce must return one score per episode"
assert any(not torch.equal(b, p) for b, p in zip(_before, _policy.parameters())), "parameters did not change — check TODO 9"
print("✅ Part 5 OK — scores of the 5 episodes:", _scores)

## Part 6 — Train the agent and plot the learning curve (slide 21)
Run the training with the hyperparameters below. It takes a few minutes on a CPU. CartPole-v1 is considered **solved** when the average return over 100 episodes reaches **475**.

In [ ]:
hyperparameters = {
    "hidden_size": 64,
    "n_episodes": 1000,
    "max_t": 500,
    "gamma": 0.99,
    "lr": 1e-3,
    "normalize": True,
}

set_seed()
env = gym.make("CartPole-v1")
env.reset(seed=SEED)
env.action_space.seed(SEED)

policy = PolicyNetwork(state_size, action_size, hyperparameters["hidden_size"]).to(device)
optimizer = optim.Adam(policy.parameters(), lr=hyperparameters["lr"])

scores = reinforce(env, policy, optimizer,
                   n_episodes=hyperparameters["n_episodes"],
                   gamma=hyperparameters["gamma"],
                   max_t=hyperparameters["max_t"],
                   normalize=hyperparameters["normalize"])

In [ ]:
def moving_average(x, w=50):
    x = np.asarray(x, dtype=float)
    return np.array([x[max(0, i - w + 1): i + 1].mean() for i in range(len(x))])

plt.figure(figsize=(10, 4.5))
plt.plot(scores, alpha=0.3, label="return of each episode")
plt.plot(moving_average(scores), linewidth=2.5, label="moving average (50 episodes)")
plt.axhline(475, linestyle="--", color="gray", label="solved threshold (475)")
plt.xlabel("Episode"); plt.ylabel("Return"); plt.ylim(0, 520)
plt.title("REINFORCE on CartPole-v1")
plt.legend(); plt.grid(alpha=0.3); plt.show()

## Part 7 — Evaluate the trained policy (slide 18)
During training the agent **samples** its actions. After training we can compare two ways of acting:
- **stochastic:** keep sampling from $\pi_\theta(\cdot \mid s)$;
- **greedy:** always take the most probable action, $\arg\max_a \pi_\theta(a \mid s)$.

### TODO 10
Complete `evaluate`: play `n_episodes` episodes **without updating** the network and return the mean and standard deviation of the returns.

> 💡 Use `torch.no_grad()`; for greedy use `probs.argmax(dim=-1).item()`; for stochastic use `Categorical(probs).sample().item()`.

In [ ]:
def evaluate(env, policy, n_episodes=20, greedy=True):
    returns = []
    for i in range(n_episodes):
        state, _ = env.reset(seed=1000 + i)
        total, done = 0.0, False
        while not done:
            # ---------------------------- TODO 10 ---------------------------
            # Choose `action` with the policy (greedy or stochastic), without gradients.
            # YOUR CODE HERE
            raise NotImplementedError("TODO 10: choose the action")
            # ----------------------------------------------------------------
            state, reward, terminated, truncated, _ = env.step(action)
            total += reward
            done = terminated or truncated
        returns.append(total)
    return float(np.mean(returns)), float(np.std(returns))

In [ ]:
eval_env = gym.make("CartPole-v1")
for mode in (True, False):
    m, s = evaluate(eval_env, policy, n_episodes=20, greedy=mode)
    print(f"{'greedy    ' if mode else 'stochastic'} policy: {m:6.1f} ± {s:5.1f}")

random_returns = []
for k in range(20):                      # baseline: a random agent on the same seeds
    eval_env.reset(seed=1000 + k)
    total, done = 0.0, False
    while not done:
        _, r, term, trunc, _ = eval_env.step(eval_env.action_space.sample())
        total += r
        done = term or trunc
    random_returns.append(total)
print(f"random policy (baseline): {np.mean(random_returns):6.1f} ± {np.std(random_returns):5.1f}")

## Part 8 — Experiments and questions
### TODO 11 — Hyperparameter experiments
`run_experiment` trains a **fresh** agent with a given configuration. Fill the list `configs` with **at least three** configurations that differ from the default (lr = 1e-3, γ = 0.99, normalize = True) in **one** thing each, for example:
- a larger learning rate (e.g. `5e-3`),
- **no** return normalization,
- a smaller discount (e.g. `γ = 0.9`).

Run the cell, look at the plot and answer the questions below. (500 episodes per configuration, to keep it short.)

In [ ]:
def run_experiment(lr=1e-3, gamma=0.99, normalize=True, n_episodes=500, seed=SEED):
    set_seed(seed)
    env = gym.make("CartPole-v1")
    env.reset(seed=seed)
    policy = PolicyNetwork(state_size, action_size).to(device)
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    return reinforce(env, policy, optimizer, n_episodes=n_episodes, gamma=gamma,
                     normalize=normalize, print_every=n_episodes)

In [ ]:
# ---------------------------- TODO 11 ---------------------------
# Each config is a dict of keyword arguments for run_experiment, plus a "label".
configs = [
    {"label": "default (lr=1e-3, γ=0.99, normalized)", "lr": 1e-3, "gamma": 0.99, "normalize": True},
    # YOUR CONFIGS HERE
]
# ----------------------------------------------------------------
assert len(configs) >= 4, "TODO 11: add at least three configurations to compare with the default"

results = {}
for cfg in configs:
    cfg = dict(cfg)
    label = cfg.pop("label")
    print(f"\n▶ {label}")
    results[label] = run_experiment(**cfg)

plt.figure(figsize=(10, 4.5))
for label, sc in results.items():
    plt.plot(moving_average(sc), linewidth=2, label=label)
plt.xlabel("Episode"); plt.ylabel("Return (moving average, 50)"); plt.ylim(0, 520)
plt.title("REINFORCE — effect of the hyperparameters"); plt.legend(); plt.grid(alpha=0.3); plt.show()

### Optional TODO 12 (bonus) — the γᵗ factor of slide 16
The update in the pseudocode is $\theta \leftarrow \theta + \alpha\,\gamma^t G_t\,\nabla_\theta \log \pi_\theta(a_t \mid s_t)$. Implement `reinforce_loss_textbook` with the extra factor $\gamma^t$, train with it (replace `reinforce_loss` temporarily or add a parameter) and compare the learning curve with the practical version.

In [ ]:
def reinforce_loss_textbook(log_probs, returns, gamma):
    # ---------------------------- TODO 12 (optional) ----------------
    # YOUR CODE HERE
    raise NotImplementedError("Optional TODO 12")
    # ----------------------------------------------------------------

### Questions
Answer briefly (2–4 sentences each) in the cell below.

**Q1.** Why does `reinforce_loss` have a **minus** sign, if REINFORCE performs gradient *ascent*?

**Q2.** REINFORCE must wait until the **end of the episode** before updating. Why? How is this different from SARSA or Q-learning?

**Q3.** Look at the learning curve of Part 6. Why is it so noisy, and why can performance **drop** after reaching 500? (slide 21)

**Q4.** Where does **exploration** come from in REINFORCE? Why don't we need ε-greedy as in DQN? (slide 18)

**Q5.** What happened with the larger learning rate and without normalization in TODO 11? Explain using the update rule.

**Q6.** Could we store old episodes in a **replay buffer** and reuse them, as in DQN? Why or why not?

**Your answers:**

- **Q1.**
- **Q2.**
- **Q3.**
- **Q4.**
- **Q5.**
- **Q6.**

---
### References
- M. Hu, *The Art of Reinforcement Learning*, Apress, 2023 — Algorithm 21 (REINFORCE) and the policy-network figure.
- Hugging Face Deep RL Course, Unit 4: *Policy Gradient with PyTorch* — https://huggingface.co/learn/deep-rl-course/unit4/
- Lecture slides: *Policy Gradient & REINFORCE* (UC2 2.4), slides 16–21.